## <font color='46B8A9'> **Caso de Estudio - Aprendizaje Supervisado**

## <font color='8EC044'> **Problema de negocio**

El sector bancario enfrenta un riesgo significativo debido a la posibilidad de que los prestatarios incumplan con el pago de sus préstamos. Este problema impacta directamente la rentabilidad y estabilidad financiera de las entidades bancarias. Para mitigar este riesgo, los bancos han comenzado a implementar modelos de aprendizaje automático que les permiten anticipar la probabilidad de incumplimiento de nuevos clientes.

En este contexto, el banco Finanzas Avanzadas ha recopilado un conjunto de datos históricos sobre sus clientes y ha contratado a un equipo de consultores para desarrollar un modelo de aprendizaje supervisado. El objetivo del análisis es identificar los factores que influyen en la probabilidad de incumplimiento de un préstamo, permitiendo al banco tomar decisiones más informadas en la concesión de créditos y reducir las pérdidas asociadas a clientes de alto riesgo.

## <font color='8EC044'> **Datos**

* **checking_account_status:** Estado de la cuenta corriente del cliente.

* **duration_months:** Duración del crédito en meses.

* **credit_history:** Historial crediticio del cliente.

* **purpose:** Propósito del crédito solicitado.

* **credit_amount:** Monto del crédito solicitado.

* **savings_account:** Estado de la cuenta de ahorros o bonos del cliente.

* **employment_since:** Antigüedad laboral del cliente.

* **installment_rate:** Porcentaje de ingresos destinados al pago del crédito.

* **other_debtors:** Presencia de otros deudores o avalistas.

* **residence_since:** Tiempo de residencia en la vivienda actual.

* **property:** Propiedades del cliente.

* **age:** Edad del cliente.

* **other_installment_plans:** Existencia de otros planes de pago.

* **housing:** Tipo de vivienda del cliente.

* **existing_credits:** Número de créditos existentes en el banco.

* **job:** Situación laboral del cliente.

* **people_liable:** Número de personas dependientes del cliente.

* **telephone:** Disponibilidad de una línea telefónica registrada a nombre del cliente.

* **foreign_worker:** Indica si el cliente es trabajador extranjero.

* **gender:** Género del cliente.

* **marital_status:** Indica si el cliente ha tenido o no un vínculo marital.

* **credit_risk:** Variable respuesta que indica si un cliente es considerado "bueno" o "malo" en términos de riesgo crediticio.

## <font color='8EC044'> **1. Inicialización**

### <font color='FF7F50'> **1.1. Librerías**

In [ ]:
import warnings
warnings.filterwarnings("ignore")

# Datos
import pandas as pd
import numpy as np

# Modelado
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score,confusion_matrix, ConfusionMatrixDisplay, f1_score
from xgboost import XGBClassifier
from sklearn.model_selection import cross_validate, StratifiedKFold, RepeatedStratifiedKFold
from sklearn.model_selection import cross_val_score
from sklearn.feature_selection import SelectFromModel
from sklearn.linear_model import Lasso, Ridge, LassoCV
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegressionCV
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
import joblib
from sklearn import tree
from sklearn.model_selection import train_test_split
from sklearn import metrics
from sklearn.ensemble import RandomForestClassifier
import graphviz
import re

### <font color='FF7F50'> **1.2. Carga de datos**

Para esta segunda entrega se decidió trabajar a partir del dataframe final procesado en la primera etapa del proyecto, el cual fue exportado y almacenado como archivo Excel en Google Drive. Esta decisión tiene como objetivo mantener la consistencia de los datos ya transformados y evitar la ejecución repetida de todo el código de preprocesamiento. De este modo, se conservan únicamente los cambios aplicados al conjunto de datos, permitiendo enfocar esta etapa exclusivamente en la implementación, comparación y análisis de distintos modelos de aprendizaje supervisado sobre una base de datos previamente estandarizada.

In [ ]:
# Descargar archivo de datos desde Google Drive
!gdown 1OU5OtW27dC9mPgKzbYpYOitWWpXzpLCl # Archivo: datos_procesados.xlsx

Downloading...
From: https://drive.google.com/uc?id=1OU5OtW27dC9mPgKzbYpYOitWWpXzpLCl
To: /content/datos_procesados.xlsx
100% 101k/101k [00:00<00:00, 50.1MB/s]


In [ ]:
# Cargar archivo de datos (tipo excel) en un DataFrame
df = pd.read_excel('datos_procesados.xlsx')
df

,checking_account_status,duration_months,credit_history,purpose,credit_amount,savings_account,employment_since,installment_rate,other_debtors,residence_since,...,other_installment_plans,housing,existing_credits,job,people_liable,telephone,foreign_worker,gender,marital_status,credit_risk
0,< 0 DM,6,Cuenta crítica/Otros créditos vigentes,Hogar,1169.0,Desconocido / Sin cuenta de ahorros,>= 7 años,4,Ninguno,4,...,Ninguno,Propia,2,Empleado calificado / Funcionario,1,"Sí, registrado a nombre del cliente",Sí,Hombre,Sin vínculo marital,Bueno
1,0 - 200 DM,48,Créditos vigentes pagados a tiempo,Hogar,5951.0,< 100 DM,1 - 4 años,2,Ninguno,2,...,Ninguno,Propia,1,Empleado calificado / Funcionario,1,Ninguno,Sí,Mujer,Con vínculo marital,Malo
2,Sin cuenta corriente,12,Cuenta crítica/Otros créditos vigentes,Educación,2096.0,< 100 DM,4 - 7 años,2,Ninguno,3,...,Ninguno,Propia,1,No calificado (residente),2,Ninguno,Sí,Hombre,Sin vínculo marital,Bueno
3,< 0 DM,42,Créditos vigentes pagados a tiempo,Hogar,7882.0,< 100 DM,4 - 7 años,2,Fiador,4,...,Ninguno,Vivienda gratuita,1,Empleado calificado / Funcionario,2,Ninguno,Sí,Hombre,Sin vínculo marital,Bueno
4,< 0 DM,24,Retrasos en pagos anteriores,Carro,4870.0,< 100 DM,1 - 4 años,3,Ninguno,4,...,Ninguno,Vivienda gratuita,2,Empleado calificado / Funcionario,2,Ninguno,Sí,Hombre,Sin vínculo marital,Malo
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,Sin cuenta corriente,12,Créditos vigentes pagados a tiempo,Hogar,1736.0,< 100 DM,4 - 7 años,3,Ninguno,4,...,Ninguno,Propia,1,No calificado (residente),1,Ninguno,Sí,Mujer,Con vínculo marital,Bueno
996,< 0 DM,30,Créditos vigentes pagados a tiempo,Carro,3857.0,< 100 DM,1 - 4 años,4,Ninguno,4,...,Ninguno,Propia,1,Gerente / Autónomo / Altamente calificado,1,"Sí, registrado a nombre del cliente",Sí,Hombre,Con vínculo marital,Bueno
997,Sin cuenta corriente,12,Créditos vigentes pagados a tiempo,Hogar,804.0,< 100 DM,>= 7 años,4,Ninguno,4,...,Ninguno,Propia,1,Empleado calificado / Funcionario,1,Ninguno,Sí,Hombre,Sin vínculo marital,Bueno
998,< 0 DM,45,Créditos vigentes pagados a tiempo,Hogar,1845.0,< 100 DM,1 - 4 años,4,Ninguno,4,...,Ninguno,Vivienda gratuita,1,Empleado calificado / Funcionario,1,"Sí, registrado a nombre del cliente",Sí,Hombre,Sin vínculo marital,Malo


## <font color='8EC044'> **2. Preparación de los datos y selección de variables**

En esta fase, se transforman las variables categóricas mediante codificación numérica, como One-Hot Encoding o Label Encoding, y se normalizan las variables numéricas para evitar sesgos en el modelo. Posteriormente, se seleccionan las variables más relevantes utilizando los métodos de selección Wrapper (RFE) e Integrado (Lasso). Ambos enfoques ayudan a reducir la complejidad del modelo, mejorando su interpretabilidad y disminuyendo el riesgo de sobreajuste. El método Wrapper evalúa el desempeño del modelo sobre diferentes subconjuntos de variables, mientras que Lasso aplica una penalización sobre los coeficientes, eliminando automáticamente aquellas variables irrelevantes.



In [ ]:
# Crear una copia del DataFrame original
df2 = df.copy()

# Mostrar información general del DataFrame y los valores únicos de cada variable categórica
df2.info()
print(" ")
for col in df2.select_dtypes(include=['object', 'category']).columns:
    print(f"{col}: {df2[col].unique()}")
print(" ")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 22 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   checking_account_status  1000 non-null   object 
 1   duration_months          1000 non-null   int64  
 2   credit_history           1000 non-null   object 
 3   purpose                  1000 non-null   object 
 4   credit_amount            1000 non-null   float64
 5   savings_account          1000 non-null   object 
 6   employment_since         1000 non-null   object 
 7   installment_rate         1000 non-null   int64  
 8   other_debtors            1000 non-null   object 
 9   residence_since          1000 non-null   int64  
 10  property                 1000 non-null   object 
 11  age                      1000 non-null   int64  
 12  other_installment_plans  1000 non-null   object 
 13  housing                  1000 non-null   object 
 14  existing_credits         

In [ ]:
df2.dtypes

,0
checking_account_status,object
duration_months,int64
credit_history,object
purpose,object
credit_amount,float64
savings_account,object
employment_since,object
installment_rate,int64
other_debtors,object
residence_since,int64


### <font color='FF7F50'> **2.1. Transformación de variables categóricas**



**Label Encoder:**

In [ ]:
# Aplicar Label Encoding a variables categóricas con solo 2 categorías
le = LabelEncoder()

# Lista de columnas categóricas a codificar
le_vars = ['gender', 'marital_status', 'foreign_worker', 'telephone', 'credit_risk']

# Aplicar LabelEncoder a cada columna
for var in le_vars:
    df2[var] = le.fit_transform(df2[var])

# Verificar
print(df2['gender'].value_counts(), '\n')
print(df2['marital_status'].value_counts(), '\n')
print(df2['foreign_worker'].value_counts(), '\n')
print(df2['telephone'].value_counts(), '\n')
print(df2['credit_risk'].value_counts())

gender
0    690
1    310
Name: count, dtype: int64 

marital_status
1    548
0    452
Name: count, dtype: int64 

foreign_worker
1    963
0     37
Name: count, dtype: int64 

telephone
0    596
1    404
Name: count, dtype: int64 

credit_risk
0    700
1    300
Name: count, dtype: int64


Dado que el modelo buscará predecir qué cliente va a incumplir con su préstamo, se conservan las categorías de la variable respuesta tal como quedaron (0 = Bueno, 1 = Malo), ya que nos interesa enfocarnos en los clientes clasificados como 'Malo', quienes serán los que incumplan.

**One-Hot Encoding**:

In [ ]:
# Lista de variables categóricas a transformar
ohe_vars = ['checking_account_status', 'credit_history', 'purpose', 'savings_account', 'employment_since', 'installment_rate',
            'other_debtors', 'residence_since', 'property', 'other_installment_plans', 'housing', 'existing_credits', 'job',
            'people_liable']

# Generar las variables dummies para las columnas seleccionadas y convertir todos los valores a tipo entero
df2 = pd.get_dummies(df2, columns=ohe_vars).astype(int)
df2.head()

,duration_months,credit_amount,age,telephone,foreign_worker,gender,marital_status,credit_risk,checking_account_status_0 - 200 DM,checking_account_status_< 0 DM,...,existing_credits_1,existing_credits_2,existing_credits_3,existing_credits_4,job_Desempleado / No calificado (no residente),job_Empleado calificado / Funcionario,job_Gerente / Autónomo / Altamente calificado,job_No calificado (residente),people_liable_1,people_liable_2
0,6,1169,67,1,1,0,1,0,0,1,...,0,1,0,0,0,1,0,0,1,0
1,48,5951,22,0,1,1,0,1,1,0,...,1,0,0,0,0,1,0,0,1,0
2,12,2096,49,0,1,0,1,0,0,0,...,1,0,0,0,0,0,0,1,0,1
3,42,7882,45,0,1,0,1,0,0,1,...,1,0,0,0,0,1,0,0,0,1
4,24,4870,53,0,1,0,1,1,0,1,...,0,1,0,0,0,1,0,0,0,1


### <font color='FF7F50'> **2.2. Normalización de variables numéricas (Min-Max Scaler)**  

In [ ]:
# Almacenar variables númericas
numcol = ['duration_months', 'credit_amount', 'age']  # Crear una lista

# Escalamiento de variable
scaler = MinMaxScaler()
for col in numcol:  # Iterar sobre las columnas núnericas
    df2[[col]] = scaler.fit_transform(df2[[col]])

df2.head()

,duration_months,credit_amount,age,telephone,foreign_worker,gender,marital_status,credit_risk,checking_account_status_0 - 200 DM,checking_account_status_< 0 DM,...,existing_credits_1,existing_credits_2,existing_credits_3,existing_credits_4,job_Desempleado / No calificado (no residente),job_Empleado calificado / Funcionario,job_Gerente / Autónomo / Altamente calificado,job_No calificado (residente),people_liable_1,people_liable_2
0,0.04,0.081041,0.979592,1,1,0,1,0,0,1,...,0,1,0,0,0,1,0,0,1,0
1,0.88,0.502734,0.061224,0,1,1,0,1,1,0,...,1,0,0,0,0,1,0,0,1,0
2,0.16,0.162787,0.612245,0,1,0,1,0,0,0,...,1,0,0,0,0,0,0,1,0,1
3,0.76,0.673016,0.530612,0,1,0,1,0,0,1,...,1,0,0,0,0,1,0,0,0,1
4,0.40,0.407407,0.693878,0,1,0,1,1,0,1,...,0,1,0,0,0,1,0,0,0,1


### <font color='FF7F50'> **2.3. Selección de variables con método Wrapper - RFE**

In [ ]:
x = df2.drop('credit_risk', axis = 1) # Variables independientes, se elimina la variable respuesta
y = df2['credit_risk'] # Variable respuesta
x.head()

,duration_months,credit_amount,age,telephone,foreign_worker,gender,marital_status,checking_account_status_0 - 200 DM,checking_account_status_< 0 DM,checking_account_status_>= 200 DM / Salario asignado,...,existing_credits_1,existing_credits_2,existing_credits_3,existing_credits_4,job_Desempleado / No calificado (no residente),job_Empleado calificado / Funcionario,job_Gerente / Autónomo / Altamente calificado,job_No calificado (residente),people_liable_1,people_liable_2
0,0.04,0.081041,0.979592,1,1,0,1,0,1,0,...,0,1,0,0,0,1,0,0,1,0
1,0.88,0.502734,0.061224,0,1,1,0,1,0,0,...,1,0,0,0,0,1,0,0,1,0
2,0.16,0.162787,0.612245,0,1,0,1,0,0,0,...,1,0,0,0,0,0,0,1,0,1
3,0.76,0.673016,0.530612,0,1,0,1,0,1,0,...,1,0,0,0,0,1,0,0,0,1
4,0.40,0.407407,0.693878,0,1,0,1,0,1,0,...,0,1,0,0,0,1,0,0,0,1


Se escoge RFE (Recursive Feature Elimination) como método de selección de variables porque permite evaluar el rendimiento del modelo al entrenarlo sobre distintos subconjuntos de variables, eliminando de forma iterativa aquellas que menos aportan. En este proceso, se utiliza LogisticRegressionCV como estimador base, el cual incorpora validación cruzada para seleccionar automáticamente el mejor nivel de regularización, mejorando así la capacidad de generalización. La métrica seleccionada para la evaluación del desempeño del modelo es recall, ya que el objetivo principal es maximizar la detección de clientes con alto riesgo de incumplimiento. En contextos con clases desbalanceadas, como este, el recall es fundamental para reducir los falsos negativos, es decir, minimizar los casos en los que el modelo no identifica correctamente a clientes realmente riesgosos.

In [ ]:
'''Este código identifica el número óptimo de variables predictoras (features) que maximizan
la precisión del modelo. Evalúa distintos valores de k (cantidad de variables seleccionadas)
y guarda el mejor resultado según la métrica de recall.

# 1. Función recursiva de selección de características
def recursive_feature_selection(X, y, model, k):
    rfe = RFE(model, n_features_to_select=k, step=1)
    fit = rfe.fit(X, y)
    X_new = fit.support_
    return X_new

# 2. Establecer estimador
model = LogisticRegressionCV(cv=5, scoring='recall', max_iter=1000)

# 3. Encontrar el mejor número de variables
scores = []
ks = range(5, x.shape[1] + 1)

for k in ks:
    x_new = recursive_feature_selection(x, y, model, k)
    x_subset = x.iloc[:, x_new]
    score = cross_val_score(model, x_subset, y, cv=5, scoring='recall').mean()
    scores.append(score)
    print(f"k={k}, recall={score:.4f}")

best_k = ks[np.argmax(scores)]
print(f"\n Mejor número de variables: {best_k}")
'''

'Este código identifica el número óptimo de variables predictoras (features) que maximizan\nla precisión del modelo. Evalúa distintos valores de k (cantidad de variables seleccionadas)\ny guarda el mejor resultado según la métrica de recall.\n\n# 1. Función recursiva de selección de características\ndef recursive_feature_selection(X, y, model, k):\n    rfe = RFE(model, n_features_to_select=k, step=1)\n    fit = rfe.fit(X, y)\n    X_new = fit.support_\n    return X_new\n\n# 2. Establecer estimador\nmodel = LogisticRegressionCV(cv=5, scoring=\'recall\', max_iter=1000)\n\n# 3. Encontrar el mejor número de variables\nscores = []\nks = range(5, x.shape[1] + 1)\n\nfor k in ks:\n    x_new = recursive_feature_selection(x, y, model, k)\n    x_subset = x.iloc[:, x_new]\n    score = cross_val_score(model, x_subset, y, cv=5, scoring=\'recall\').mean()\n    scores.append(score)\n    print(f"k={k}, recall={score:.4f}")\n\nbest_k = ks[np.argmax(scores)]\nprint(f"\n Mejor número de variables: {best_k}

Con el código anterior, se identificó que el mejor número de variables
𝑘 a seleccionar es 50.

In [ ]:
# Función recursiva de selección de características
def recursive_feature_selection(X, y, model, k):  # model=modelo que me va a servir de estimador
    rfe = RFE(model, n_features_to_select=k, step=1)  # step=1 cada cuanto el toma la decision de tomar una caracteristica
    fit = rfe.fit(X, y)
    X_new = fit.support_
    print("Num Features: %s" % (fit.n_features_))
    print("Selected Features: %s" % (fit.support_))
    print("Feature Ranking: %s" % (fit.ranking_))
    return X_new

In [ ]:
# Establecer Estimador
model = LogisticRegressionCV() # Modelo para una variable respuesta de tipo categórica (clasificación)

# Obtener columnas seleccionadas
X_new = recursive_feature_selection(x, y, model, 50)

# Nuevo conjunto de datos
df_wrapper = x.iloc[:,X_new]
df_wrapper.head()

Num Features: 50
Selected Features: [ True  True  True  True  True  True  True  True  True  True  True False
  True False  True  True  True  True  True  True  True  True  True  True
  True  True False  True  True  True  True  True  True False  True  True
  True  True  True  True False  True False  True  True  True  True  True
 False  True  True  True  True False False  True  True  True False False
  True False]
Feature Ranking: [ 1  1  1  1  1  1  1  1  1  1  1  4  1  2  1  1  1  1  1  1  1  1  1  1
  1  1  7  1  1  1  1  1  1  9  1  1  1  1  1  1  6  1  3  1  1  1  1  1
  5  1  1  1  1 11  8  1  1  1 12 13  1 10]


,duration_months,credit_amount,age,telephone,foreign_worker,gender,marital_status,checking_account_status_0 - 200 DM,checking_account_status_< 0 DM,checking_account_status_>= 200 DM / Salario asignado,...,other_installment_plans_Banco,other_installment_plans_Ninguno,housing_Alquiler,housing_Propia,housing_Vivienda gratuita,existing_credits_1,existing_credits_4,job_Desempleado / No calificado (no residente),job_Empleado calificado / Funcionario,people_liable_1
0,0.04,0.081041,0.979592,1,1,0,1,0,1,0,...,0,1,0,1,0,0,0,0,1,1
1,0.88,0.502734,0.061224,0,1,1,0,1,0,0,...,0,1,0,1,0,1,0,0,1,1
2,0.16,0.162787,0.612245,0,1,0,1,0,0,0,...,0,1,0,1,0,1,0,0,0,0
3,0.76,0.673016,0.530612,0,1,0,1,0,1,0,...,0,1,0,0,1,1,0,0,1,0
4,0.40,0.407407,0.693878,0,1,0,1,0,1,0,...,0,1,0,0,1,0,0,0,1,0


In [ ]:
list(df_wrapper.columns)

['duration_months',
 'credit_amount',
 'age',
 'telephone',
 'foreign_worker',
 'gender',
 'marital_status',
 'checking_account_status_0 - 200 DM',
 'checking_account_status_< 0 DM',
 'checking_account_status_>= 200 DM / Salario asignado',
 'checking_account_status_Sin cuenta corriente',
 'credit_history_Cuenta crítica/Otros créditos vigentes',
 'credit_history_Sin créditos/Todos pagados',
 'credit_history_Todos los créditos en este banco pagados',
 'purpose_Carro',
 'purpose_Educación',
 'purpose_Hogar',
 'purpose_Negocios',
 'purpose_Otro',
 'savings_account_100 - 500 DM',
 'savings_account_500 - 1000 DM',
 'savings_account_< 100 DM',
 'savings_account_>= 1000 DM',
 'savings_account_Desconocido / Sin cuenta de ahorros',
 'employment_since_4 - 7 años',
 'employment_since_< 1 año',
 'employment_since_>= 7 años',
 'employment_since_Desempleado',
 'installment_rate_1',
 'installment_rate_2',
 'installment_rate_4',
 'other_debtors_Co-solicitante',
 'other_debtors_Fiador',
 'other_debtors_

### <font color='FF7F50'> **2.4. Selección de variables con método integrado Lasso**

Se escoge LassoCV como método de selección de variables porque permite identificar automáticamente el mejor valor de penalización (alpha) mediante validación cruzada, lo que facilita la eliminación de variables irrelevantes y mejora la interpretabilidad del modelo. Posteriormente, se evalúan distintos valores de variables máximas a seleccionar (max_features) para encontrar el subconjunto óptimo que maximiza el rendimiento del modelo. En este caso, se utiliza recall como métrica de evaluación durante la validación cruzada con LogisticRegressionCV, ya que el objetivo principal es maximizar la detección de clientes con alto riesgo de incumplimiento.

In [ ]:
'''
Este script selecciona variables usando LassoCV, que encuentra automáticamente el mejor alpha.
Evalúa distintos valores de max_features (número de variables) para identificar el subconjunto
que maximiza la precisión (recall) en clasificación con validación cruzada.

# 1. Ajustar LassoCV una sola vez (con validación cruzada para encontrar el mejor alpha)
lasso_cv = LassoCV(cv=5, max_iter=10000)
lasso_cv.fit(x, y)
print(f"Mejor alpha encontrado: {lasso_cv.alpha_} \n")

# 2. Probar diferentes valores de max_features
scores = []
feature_range = range(5, x.shape[1] + 1)

for k in feature_range:
    selector = SelectFromModel(lasso_cv, max_features=k, threshold=-np.inf)  # -inf para forzar selección hasta el máximo
    selector.fit(x, y)
    x_subset = x.loc[:, selector.get_support()]

    # Validación cruzada con regresión logística
    from sklearn.linear_model import LogisticRegressionCV
    model = LogisticRegressionCV(cv=5, scoring='recall', max_iter=1000)

    score = cross_val_score(model, x_subset, y, cv=5, scoring='recall').mean()
    scores.append(score)
    print(f"max_features={k}, recall={score:.4f}")

# 3. Mostrar el mejor número de variables
best_k = feature_range[np.argmax(scores)]
print(f"\n Mejor número de variables con Lasso: {best_k} (recall = {max(scores):.4f})")

# 4. Mostrar los nombres de las variables seleccionadas con el mejor k
selector_final = SelectFromModel(lasso_cv, max_features=best_k, threshold=-np.inf)
selector_final.fit(x, y)
selected_vars = x.columns[selector_final.get_support()]
print(f"\n Variables seleccionadas con Lasso ({best_k}):")
print(selected_vars.tolist())
'''

'\nEste script selecciona variables usando LassoCV, que encuentra automáticamente el mejor alpha.\nEvalúa distintos valores de max_features (número de variables) para identificar el subconjunto\nque maximiza la precisión (recall) en clasificación con validación cruzada.\n\n# 1. Ajustar LassoCV una sola vez (con validación cruzada para encontrar el mejor alpha)\nlasso_cv = LassoCV(cv=5, max_iter=10000)\nlasso_cv.fit(x, y)\nprint(f"Mejor alpha encontrado: {lasso_cv.alpha_} \n")\n\n# 2. Probar diferentes valores de max_features\nscores = []\nfeature_range = range(5, x.shape[1] + 1)\n\nfor k in feature_range:\n    selector = SelectFromModel(lasso_cv, max_features=k, threshold=-np.inf)  # -inf para forzar selección hasta el máximo\n    selector.fit(x, y)\n    x_subset = x.loc[:, selector.get_support()]\n\n    # Validación cruzada con regresión logística\n    from sklearn.linear_model import LogisticRegressionCV\n    model = LogisticRegressionCV(cv=5, scoring=\'recall\', max_iter=1000)  # AQ

Con el código anterior, se identificó que el mejor valor de alpha encontrado es 0.002914726460706714, y que el número óptimo de variables a seleccionar es 57. Estos valores se utilizan a continuación para ajustar un modelo Lasso definitivo y generar el subconjunto final de variables relevantes.

In [ ]:
# Selector de variables con Lasso (usando el alpha ya encontrado)
sel_ = SelectFromModel(Lasso(alpha=0.002914726460706714, max_iter=10000), max_features=57)
sel_.fit(x, y)
print(sel_.estimator_.coef_)

# Obtener variables seleccionadas
X_new = sel_.get_support()

# Nuevo DataFrame con variables seleccionadas
df_lasso = x.iloc[:, X_new]
df_lasso.head()

[ 0.27005985  0.         -0.         -0.0277852   0.06066182  0.01054273
 -0.04974421  0.07480984  0.16378394 -0.         -0.12537579 -0.
 -0.10090944 -0.          0.09750352  0.08889288  0.          0.00764413
 -0.04117114 -0.         -0.          0.04021027 -0.          0.08930527
 -0.00737449 -0.02204395  0.         -0.04600649  0.04701134 -0.0065995
  0.         -0.00514675 -0.          0.          0.05618254  0.02111065
 -0.11423561 -0.         -0.0241729   0.05192093  0.         -0.
  0.         -0.03935482 -0.          0.07452058  0.         -0.06103623
  0.          0.05647969 -0.         -0.         -0.01991472  0.
 -0.          0.         -0.          0.         -0.         -0.
 -0.          0.        ]


,duration_months,telephone,foreign_worker,gender,marital_status,checking_account_status_0 - 200 DM,checking_account_status_< 0 DM,checking_account_status_Sin cuenta corriente,credit_history_Cuenta crítica/Otros créditos vigentes,credit_history_Sin créditos/Todos pagados,...,installment_rate_4,other_debtors_Co-solicitante,other_debtors_Fiador,residence_since_1,residence_since_2,property_Bienes raíces,property_Desconocido / Sin propiedad,other_installment_plans_Ninguno,housing_Alquiler,existing_credits_1
0,0.04,1,1,0,1,0,1,0,1,0,...,1,0,0,0,0,1,0,1,0,0
1,0.88,0,1,1,0,1,0,0,0,0,...,0,0,0,0,1,1,0,1,0,1
2,0.16,0,1,0,1,0,0,1,1,0,...,0,0,0,0,0,1,0,1,0,1
3,0.76,0,1,0,1,0,1,0,0,0,...,0,0,1,0,0,0,0,1,0,1
4,0.40,0,1,0,1,0,1,0,0,0,...,0,0,0,0,0,0,1,1,0,0


In [ ]:
list(df_lasso.columns)

['duration_months',
 'telephone',
 'foreign_worker',
 'gender',
 'marital_status',
 'checking_account_status_0 - 200 DM',
 'checking_account_status_< 0 DM',
 'checking_account_status_Sin cuenta corriente',
 'credit_history_Cuenta crítica/Otros créditos vigentes',
 'credit_history_Sin créditos/Todos pagados',
 'credit_history_Todos los créditos en este banco pagados',
 'purpose_Educación',
 'purpose_Hogar',
 'savings_account_100 - 500 DM',
 'savings_account_< 100 DM',
 'savings_account_>= 1000 DM',
 'savings_account_Desconocido / Sin cuenta de ahorros',
 'employment_since_4 - 7 años',
 'employment_since_< 1 año',
 'employment_since_>= 7 años',
 'installment_rate_1',
 'installment_rate_4',
 'other_debtors_Co-solicitante',
 'other_debtors_Fiador',
 'residence_since_1',
 'residence_since_2',
 'property_Bienes raíces',
 'property_Desconocido / Sin propiedad',
 'other_installment_plans_Ninguno',
 'housing_Alquiler',
 'existing_credits_1']

Tras aplicar ambas metodologías de selección de variables, se eligió el método Lasso sobre RFE para la construcción de los modelos de predicción. Con RFE, se seleccionaron 50 variables de un total de 62, pero los resultados obtenidos con el método Lasso, que seleccionó solo 31 variables, demostraron un mejor rendimiento. Lasso, al aplicar una penalización sobre los coeficientes, no solo eliminó variables irrelevantes, sino que también ayudó a reducir la complejidad del modelo, mejorando la capacidad de generalización y minimizando el riesgo de sobreajuste. Además, Lasso permite mantener la interpretabilidad del modelo, ya que al seleccionar un número más reducido de variables, facilita la comprensión de qué factores son más relevantes para la predicción. La combinación de un modelo más sencillo y de mejor desempeño en términos de precisión y capacidad predictiva justifica la elección de las 31 variables seleccionadas por Lasso para la modelización final.

### <font color='FF7F50'> **2.5. Definición de función de evaluación del modelo mediante validación cruzada**

In [ ]:
# Definición de función de validación cruzada con todas las métricas (para clasificación)
def cross_validation(model, X, y, cv=5):
    """Función para realizar validación cruzada y devolver métricas de rendimiento."""
    _scoring = ['accuracy', 'precision', 'recall', 'f1']
    results = cross_validate(estimator=model,
                             X=X,
                             y=y,
                             cv=cv,
                             scoring=_scoring,
                             return_train_score=True)

    return {
        "Training Accuracy scores": results['train_accuracy'],
        "Mean Training Accuracy": results['train_accuracy'].mean() * 100,
        "Training Precision scores": results['train_precision'],
        "Mean Training Precision": results['train_precision'].mean(),
        "Training Recall scores": results['train_recall'],
        "Mean Training Recall": results['train_recall'].mean(),
        "Training F1 scores": results['train_f1'],
        "Mean Training F1 Score": results['train_f1'].mean(),
        "Validation Accuracy scores": results['test_accuracy'],
        "Mean Validation Accuracy": results['test_accuracy'].mean() * 100,
        "Validation Precision scores": results['test_precision'],
        "Mean Validation Precision": results['test_precision'].mean(),
        "Validation Recall scores": results['test_recall'],
        "Mean Validation Recall": results['test_recall'].mean(),
        "Validation F1 scores": results['test_f1'],
        "Mean Validation F1 Score": results['test_f1'].mean()
    }

## <font color='8EC044'> **3. Aplicación de algoritmo XGBoost**

### <font color='FF7F50'> **3.1. XGBoost base**

In [ ]:
# Nombres de columnas de X sean válidos para XGBoost
df_lasso.columns = [f"var_{i}" for i in range(df_lasso.shape[1])]

X = df_lasso
y = df2['credit_risk']

# Instancia del modelo
bst = XGBClassifier(objective='binary:logistic', random_state=123)

# Ajustar el modelo a los datos de entrenamiento
bst.fit(X, y)

# Predicciones
preds = bst.predict(X)

# Ejecutar la función personalizada de cross_validation (clasificación)
model_results = cross_validation(bst, X, y, 5)

# Resultados de las métricas de cross_validation
print("\nMean Validation Accuracy: ", model_results['Mean Validation Accuracy'],
      "\nMean Validation Precision: ", model_results['Mean Validation Precision'],
      "\nMean Validation Recall: ", model_results['Mean Validation Recall'],
      "\nMean Validation F1 Score: ", model_results['Mean Validation F1 Score'])


Mean Validation Accuracy:  72.8 
Mean Validation Precision:  0.5514110230635655 
Mean Validation Recall:  0.5 
Mean Validation F1 Score:  0.5238397519506501


In [ ]:
model_results

{'Training Accuracy scores': array([1.     , 0.99875, 0.99625, 0.99875, 0.99875]),
 'Mean Training Accuracy': np.float64(99.85000000000002),
 'Training Precision scores': array([1.       , 1.       , 0.9958159, 1.       , 1.       ]),
 'Mean Training Precision': np.float64(0.999163179916318),
 'Training Recall scores': array([1.        , 0.99583333, 0.99166667, 0.99583333, 0.99583333]),
 'Mean Training Recall': np.float64(0.9958333333333333),
 'Training F1 scores': array([1.        , 0.99791232, 0.99373695, 0.99791232, 0.99791232]),
 'Mean Training F1 Score': np.float64(0.9974947807933194),
 'Validation Accuracy scores': array([0.735, 0.69 , 0.73 , 0.73 , 0.755]),
 'Mean Validation Accuracy': np.float64(72.8),
 'Validation Precision scores': array([0.56363636, 0.48214286, 0.5625    , 0.55555556, 0.59322034]),
 'Mean Validation Precision': np.float64(0.5514110230635655),
 'Validation Recall scores': array([0.51666667, 0.45      , 0.45      , 0.5       , 0.58333333]),
 'Mean Validation R

### <font color='FF7F50'> **3.2. Selección y optimización de hiperparámetros: Búsqueda por cuadrícula**

Con el fin de mejorar el rendimiento del modelo de clasificación mediante XGBoost, se procede a implementar un proceso de búsqueda por cuadrícula (Grid Search) para seleccionar los mejores hiperparámetros. Esta técnica permite evaluar combinaciones específicas de parámetros y determinar cuáles producen los mejores resultados en validación cruzada.

**Hiperparámetros seleccionados:** Se ha determinado que los hiperparámetros que más influencia tienen en el rendimiento de los modelos basados en árboles de decisión como XGBoost son:

* **max_depth:** La profundidad máxima es un parámetro frecuente entre algoritmos basados en árboles de decisión. Este tiene como meta controlar el máximo de divisiones o ramificaciones hasta las que puede bajar el árbol. Los valores más frecuentes que puede tomar están en el rango de 3-6, usando profundidades de 3 en modelos complejos pero que se desean explicar de manera sencilla y sin mucho sobreajuste. Mientras que los valores de 6 o mayores son porque se desea capturar interacciones más complejas, arriesgándose a sobreajustes.

* **learning_rate:** El rate de aprendizaje en los modelos de machine learning hace referencia a la velocidad de aprendizaje del modelo. Este tradicionalmente toma valores de 0-1, los cuales se suelen asignar dependiendo de qué tan complejo sea el problema. Para este caso, se utilizan los valores: 1, 0.1, 0.01, 0.001, los cuales permiten cubrir un espectro amplio, desde aprendizaje rápido hasta refinamiento progresivo.

* **subsample:** El subsample es el encargado de determinar con cuántos datos son entrenados los árboles en cada iteración. Este usualmente se usa para disminuir la posibilidad de tener sobreajustes y ayuda a reducir el tiempo que tarda en entrenar el modelo al no usar en cada árbol el 100% de los datos. Este comúnmente suele manejarse en rangos de 0.5-1, números que representan el porcentaje que dedicará el modelo de los datos para entrenar cada árbol.

**Referencias:**

https://medium.com/@rithpansanga/optimizing-xgboost-a-guide-to-hyperparameter-tuning-77b6e48e289d

https://blog.dataiku.com/narrowing-the-search-which-hyperparameters-really-matter


In [ ]:
# Definir la cuadrícula de hiperparámetros
param_grid = {
    'max_depth': [3, 4, 5, 6, 7],
    'learning_rate': [1, 0.1, 0.01, 0.001],
    'subsample': [0.5, 0.6, 0.7, 0.8, 0.9, 1]
}

# Crear el modelo XGBoost para clasificación
modeloH = XGBClassifier(random_state=123)

# Crear el objeto GridSearchCV
grid_search = GridSearchCV(modeloH, param_grid, cv=5, scoring='recall')

# Ajustar la búsqueda a los datos
grid_search.fit(X, y)

# Mostrar los mejores hiperparámetros y su puntaje asociado
print("Mejores hiperparámetros encontrados: ", grid_search.best_params_)
print("Mejor puntuación (recall promedio): ", grid_search.best_score_)

Mejores hiperparámetros encontrados:  {'learning_rate': 1, 'max_depth': 3, 'subsample': 0.5}
Mejor puntuación (recall promedio):  0.5633333333333332


La elección de recall como métrica para la búsqueda de hiperparámetros mediante GridSearchCV se justifica plenamente en un contexto de clases desbalanceadas, donde la prioridad del negocio es detectar la mayor cantidad posible de clientes en riesgo de incumplimiento (clase minoritaria). Al seleccionar scoring='recall', estamos enfocando el modelo en minimizar los falsos negativos, es decir, en reducir los casos en los que un cliente riesgoso no es identificado como tal.

En este caso donde la clase minoritaria representa solo el 30% de las observaciones, métricas como la accuracy pueden resultar engañosas, ya que un modelo que predice en su mayoría la clase mayoritaria puede mostrar un rendimiento aparentemente alto, pero fallar gravemente en la detección de los casos realmente importantes. Optimizar por recall permite construir modelos más sensibles, capaces de identificar correctamente a la mayoría de los clientes de alto riesgo, aunque esto implique aceptar un mayor número de falsos positivos, lo cual puede ser una decisión estratégica razonable dependiendo del costo asociado a cada tipo de error.

### <font color='FF7F50'> **3.3. XGBoost con hiperparámetros optimizados**

In [ ]:
# modelo XGBClassifier con mejores hiperparámetros
bst = XGBClassifier(**grid_search.best_params_, objective='binary:logistic', random_state=123)

# Ajustar el modelo a los datos de entrenamiento
bst.fit(X, y)

# Predicciones
preds = bst.predict(X)

# Función de cross_validation (clasificación)
model_results_opti = cross_validation(bst, X, y, 5)

# Resultados de las métricas de cross_validation
print("\nMean Validation Accuracy: ", model_results_opti['Mean Validation Accuracy'],
      "\nMean Validation Precision: ", model_results_opti['Mean Validation Precision'],
      "\nMean Validation Recall: ", model_results_opti['Mean Validation Recall'],
      "\nMean Validation F1 Score: ", model_results_opti['Mean Validation F1 Score'])


Mean Validation Accuracy:  72.2 
Mean Validation Precision:  0.5354047831253713 
Mean Validation Recall:  0.5633333333333332 
Mean Validation F1 Score:  0.5483328565265717


### <font color='FF7F50'> **3.4. Resultados**

In [ ]:
# Resultados del modelo base
acc_base = 0.728
prec_base = 0.5514110230635655
recall_base = 0.5
f1_base = 0.5238397519506501

# Resultados del modelo optimizado
acc_opt = 0.722
prec_opt = 0.5354047831253713
recall_opt = 0.563333333333333
f1_opt = 0.5483328565265717

# Crear tabla de comparación
results_clf = pd.DataFrame([
    ['XGBoost base', acc_base, prec_base, recall_base, f1_base],
    ['XGBoost optimizado', acc_opt, prec_opt, recall_opt, f1_opt]
], columns=['Modelo', 'Accuracy', 'Precision', 'Recall', 'F1-Score'])

# Mostrar tabla
results_clf

,Modelo,Accuracy,Precision,Recall,F1-Score
0,XGBoost base,0.728,0.551411,0.500000,0.523840
1,XGBoost optimizado,0.722,0.535405,0.563333,0.548333


**Comparación entre XGBoost base y optimizado:**

* **Accuracy:** El modelo base tiene un mejor desempeño general con un accuracy de 72.8%, frente al 72.2% del optimizado. Esto sugiere que, en términos globales, el modelo base acierta más en sus predicciones.

* **Precision:** También es superior en el modelo base (0.551 vs. 0.535), lo que indica que cuando predice que un cliente es riesgoso, tiene una mayor proporción de aciertos.

* **Recall:** El modelo optimizado mejora levemente el recall (0.563 vs. 0.50), lo que significa que detecta ligeramente más clientes realmente riesgosos, aunque la diferencia es mínima (~6.3 puntos porcentuales).

* **F1-Score:** El modelo optimizado obtiene un F1-Score un poco más alto (0.548 vs. 0.523), lo que refleja un mejor balance entre precisión y recall.

Aunque el objetivo de optimizar hiperparámetros suele ser mejorar el desempeño del modelo, en este caso, el modelo XGBoost optimizado no supera al modelo base de forma significativa en ninguna métrica clave. La mejora en recall es notable, pero la diferencia en las demás métricas (accuracy, precision y F1) es más favorable al modelo base. Por tanto, si se desea mantener un mejor equilibrio general en el desempeño, el modelo XGBoost base sería preferible.

## <font color='8EC044'> **4. Aplicación de algoritmo Random Forest**

### <font color='FF7F50'> **4.1. Random Forest base**

In [ ]:
#  Entrenamiento del modelo de Random Forest sin hiperparametros
rf_model = RandomForestClassifier(random_state = 123)

In [ ]:
# Realizar validación cruzada con el conjunto de características seleccionadas
cv_results = cross_validation(rf_model, X, y, 5)

# Resultados de las métricas de cross_validation
print("\nMean Validation Accuracy: ", cv_results['Mean Validation Accuracy'],
      "\nMean Validation Precision: ", cv_results['Mean Validation Precision'],
      "\nMean Validation Recall: ", cv_results['Mean Validation Recall'],
      "\nMean Validation F1 Score: ", cv_results['Mean Validation F1 Score'])


Mean Validation Accuracy:  76.8 
Mean Validation Precision:  0.6682051282051282 
Mean Validation Recall:  0.45 
Mean Validation F1 Score:  0.536205185495508


In [ ]:
cv_results

{'Training Accuracy scores': array([1., 1., 1., 1., 1.]),
 'Mean Training Accuracy': np.float64(100.0),
 'Training Precision scores': array([1., 1., 1., 1., 1.]),
 'Mean Training Precision': np.float64(1.0),
 'Training Recall scores': array([1., 1., 1., 1., 1.]),
 'Mean Training Recall': np.float64(1.0),
 'Training F1 scores': array([1., 1., 1., 1., 1.]),
 'Mean Training F1 Score': np.float64(1.0),
 'Validation Accuracy scores': array([0.775, 0.755, 0.755, 0.78 , 0.775]),
 'Mean Validation Accuracy': np.float64(76.8),
 'Validation Precision scores': array([0.66666667, 0.64102564, 0.66666667, 0.7       , 0.66666667]),
 'Mean Validation Precision': np.float64(0.6682051282051282),
 'Validation Recall scores': array([0.5       , 0.41666667, 0.36666667, 0.46666667, 0.5       ]),
 'Mean Validation Recall': np.float64(0.45),
 'Validation F1 scores': array([0.57142857, 0.50505051, 0.47311828, 0.56      , 0.57142857]),
 'Mean Validation F1 Score': np.float64(0.536205185495508)}

### <font color='FF7F50'> **4.2. Selección y optimización de hiperparámetros: Búsqueda por cuadrícula**

Dado que el conjunto de datos presenta clases desbalanceadas, se definió un grid de hiperparámetros que permite controlar el sobreajuste, mejorar la capacidad de generalización y atender adecuadamente la clase minoritaria. A continuación se resume la razón detrás de cada parámetro:

* **n_estimators:** Aumentar el número de árboles mejora la estabilidad del modelo, aunque con mayor costo computacional. Se probaron valores entre 50 y 200.

* **max_depth:** Limita la complejidad de los árboles. Se evaluaron distintas profundidades (5, 10, 20, None) para evitar tanto el subajuste como el sobreajuste.

* **min_samples_split y min_samples_leaf:** Controlan cuándo dividir nodos y el tamaño mínimo de hojas. Se probaron valores moderados para evitar ramas muy específicas que no generalicen bien.

* **max_features:** Define cuántas variables se usan al dividir. Distintas opciones ('sqrt', 'log2', 'auto', None) permiten explorar distintos niveles de aleatoriedad y reducir la varianza.

* **class_weight='balanced':** Esencial en contextos con desbalance, ya que ajusta automáticamente los pesos de cada clase para mejorar métricas como recall y F1-score en la clase minoritaria.

**Referencia:**

https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html

In [ ]:
# Definicion de hiperparámetros para bosques aleatorios
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [5, 10, 20, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4, 8],
    'max_features': ['sqrt', 'log2', 'auto', None],
    'class_weight': ['balanced']  # Para manejar el desbalanceo de clases
}

In [ ]:
# Crear el modelo de Random Forest
rf_model = RandomForestClassifier(random_state = 123)

In [ ]:
# Definir el GridSearchCV con validación cruzada
grid_search = GridSearchCV(estimator=rf_model,
                           param_grid=param_grid,
                           cv=5,  # Número de folds para validación cruzada
                           scoring='recall',
                           n_jobs=-1,  # Usar todos los núcleos disponibles para acelerar la búsqueda
                           verbose=2)

En este caso, se utilizó el recall como métrica en GridSearchCV debido a que el objetivo principal es maximizar la detección de clientes en riesgo de incumplimiento. En contextos de clases desbalanceadas, como este, donde la clase minoritaria representa un porcentaje reducido de los datos, es fundamental reducir la cantidad de falsos negativos.

In [ ]:
# Ajustar el modelo al conjunto de entrenamiento
grid_search.fit(X, y)

Fitting 5 folds for each of 576 candidates, totalling 2880 fits


GridSearchCV(cv=5, estimator=RandomForestClassifier(random_state=123),
             n_jobs=-1,
             param_grid={'class_weight': ['balanced'],
                         'max_depth': [5, 10, 20, None],
                         'max_features': ['sqrt', 'log2', 'auto', None],
                         'min_samples_leaf': [1, 2, 4, 8],
                         'min_samples_split': [2, 5, 10],
                         'n_estimators': [50, 100, 200]},
             scoring='recall', verbose=2)

In [ ]:
# Mostrar los mejores hiperparámetros y su puntaje asociado
print("Mejores hiperparámetros encontrados: ", grid_search.best_params_)
print("Mejor puntuación (recall promedio): ", grid_search.best_score_)

Mejores hiperparámetros encontrados:  {'class_weight': 'balanced', 'max_depth': 5, 'max_features': None, 'min_samples_leaf': 8, 'min_samples_split': 2, 'n_estimators': 200}
Mejor puntuación (recall promedio):  0.7833333333333333


### <font color='FF7F50'> **4.3. Random Forest con hiperparámetros optimizados**

In [ ]:
# Obtener el mejor modelo
best_rf_model = grid_search.best_estimator_
best_rf_model

RandomForestClassifier(class_weight='balanced', max_depth=5, max_features=None,
                       min_samples_leaf=8, n_estimators=200, random_state=123)

In [ ]:
# Realizar validación cruzada con el modelo optimizado
cv_results_optimized = cross_validation(best_rf_model, X, y, 5)

# Resultados de las métricas de cross_validation
print("\nMean Validation Accuracy: ", cv_results_optimized['Mean Validation Accuracy'],
      "\nMean Validation Precision: ", cv_results_optimized['Mean Validation Precision'],
      "\nMean Validation Recall: ", cv_results_optimized['Mean Validation Recall'],
      "\nMean Validation F1 Score: ", cv_results_optimized['Mean Validation F1 Score'])


Mean Validation Accuracy:  69.9 
Mean Validation Precision:  0.5006271185764425 
Mean Validation Recall:  0.7833333333333333 
Mean Validation F1 Score:  0.6103504867867339


In [ ]:
cv_results_optimized

{'Training Accuracy scores': array([0.745  , 0.76125, 0.74875, 0.75   , 0.72625]),
 'Mean Training Accuracy': np.float64(74.625),
 'Training Precision scores': array([0.55      , 0.56786704, 0.55342466, 0.55464481, 0.52658228]),
 'Mean Training Precision': np.float64(0.5505037561539018),
 'Training Recall scores': array([0.825     , 0.85416667, 0.84166667, 0.84583333, 0.86666667]),
 'Mean Training Recall': np.float64(0.8466666666666667),
 'Training F1 scores': array([0.66      , 0.68219634, 0.6677686 , 0.669967  , 0.65511811]),
 'Mean Training F1 Score': np.float64(0.6670100082822977),
 'Validation Accuracy scores': array([0.665, 0.7  , 0.755, 0.69 , 0.685]),
 'Mean Validation Accuracy': np.float64(69.9),
 'Validation Precision scores': array([0.46601942, 0.5       , 0.56321839, 0.4893617 , 0.48453608]),
 'Mean Validation Precision': np.float64(0.5006271185764425),
 'Validation Recall scores': array([0.8       , 0.75      , 0.81666667, 0.76666667, 0.78333333]),
 'Mean Validation Recall

### <font color='FF7F50'> **4.4. Resultados**

In [ ]:
# Resultados del modelo base
acc_base2 = 0.768
prec_base2 = 0.6682051282051282
recall_base2 = 0.45
f1_base2 = 0.536205185495508

# Resultados del modelo optimizado
acc_opt2 = 0.699
prec_opt2 = 0.5006271185764425
recall_opt2 = 0.7833333333333333
f1_opt2 = 0.6103504867867339

# Crear tabla de comparación
results_clf2 = pd.DataFrame([
    ['Random Forest base', acc_base2, prec_base2, recall_base2, f1_base2],
    ['Random Forest optimizado', acc_opt2, prec_opt2, recall_opt2, f1_opt2]
], columns=['Modelo', 'Accuracy', 'Precision', 'Recall', 'F1-Score'])

# Mostrar tabla
results_clf2

,Modelo,Accuracy,Precision,Recall,F1-Score
0,Random Forest base,0.768,0.668205,0.450000,0.536205
1,Random Forest optimizado,0.699,0.500627,0.783333,0.610350


**Comparación entre Random Forest base y optimizado:**

* **Accuracy:** El modelo base presenta un accuracy superior (76.8% frente a 69.9%), lo que sugiere que en términos generales comete menos errores de clasificación.

* **Precision:** También es más alto en el modelo base (0.668 vs. 0.50), lo que significa que tiene menos falsos positivos cuando predice clientes riesgosos.

* **Recall:** Aquí es donde el modelo optimizado se destaca notablemente (0.783 frente a 0.450). Detecta muchos más clientes que realmente son riesgosos, reduciendo significativamente los falsos negativos.

* **F1-Score:** El modelo optimizado alcanza un F1-Score de 0.6104, claramente superior al del modelo base (0.536), lo que indica un mejor equilibrio general entre precisión y recall.

El Random Forest optimizado sacrifica precisión y accuracy para mejorar sustancialmente el recall, lo cual es altamente deseable en este escenario donde el costo de no identificar a un cliente riesgoso es alto. Aunque clasifica correctamente a menos clientes en total (menor accuracy), detecta a la gran mayoría de los clientes realmente riesgosos, lo que se refleja en su F1-Score, el más alto entre todos los modelos evaluados hasta ahora.

## <font color='8EC044'> **5. Comparación y conclusiones sobre los modelos implementados**

En esta sección, se presentan los resultados obtenidos de los diferentes modelos implementados para la clasificación del riesgo crediticio. Se comparan tanto los modelos base como aquellos optimizados mediante técnicas de ajuste de hiperparámetros, con el objetivo de evaluar su rendimiento en términos de precisión, recall, accuracy y F1-Score. Además, se incluye el análisis de la Regresión Logística, modelo utilizado en la entrega 1 del proyecto, para ofrecer una visión integral de los enfoques explorados. Esta comparación permite identificar el modelo más adecuado para el contexto de predicción del riesgo crediticio, considerando tanto la capacidad de detección de clientes en riesgo como la precisión en la clasificación.

In [ ]:
# Resultados del modelo base de Regresión logística
acc = 0.76
prec = 0.7058823529411765
recall = 0.48
f1 = 0.5714285714285714

# Crear tabla de comparación
results_clf3 = pd.DataFrame([
    ['Regresión logística', acc, prec, recall, f1]
], columns=['Modelo', 'Accuracy', 'Precision', 'Recall', 'F1-Score'])

# Mostrar tabla
results_clf3

,Modelo,Accuracy,Precision,Recall,F1-Score
0,Regresión logística,0.76,0.705882,0.48,0.571429


In [ ]:
# Unir todas las tablas en una sola
results_total = pd.concat([results_clf, results_clf2, results_clf3], ignore_index=True)

# Mostrar la tabla unificada
results_total

,Modelo,Accuracy,Precision,Recall,F1-Score
0,XGBoost base,0.728,0.551411,0.500000,0.523840
1,XGBoost optimizado,0.722,0.535405,0.563333,0.548333
2,Random Forest base,0.768,0.668205,0.450000,0.536205
3,Random Forest optimizado,0.699,0.500627,0.783333,0.610350
4,Regresión logística,0.760,0.705882,0.480000,0.571429


**Conclusiones:**

El análisis de los modelos indica que, si bien varios algoritmos ofrecen buenos desempeños en diferentes métricas, la elección del modelo más adecuado depende directamente del objetivo principal del proyecto: detectar de forma efectiva a los clientes con alto riesgo de incumplimiento de pagos. En este contexto, métricas como recall y F1-Score son prioritarias, ya que reflejan la capacidad del modelo para identificar correctamente a los clientes riesgosos sin comprometer demasiado la precisión.

El modelo Random Forest optimizado se posiciona como el más adecuado para este objetivo, ya que alcanza el mayor recall (0.783), lo que indica una excelente capacidad para identificar a clientes con alto riesgo. A pesar de que su precisión (0.5006) y accuracy (0.699) son inferiores a otros modelos, su F1-Score de 0.6104, el más alto de todos, revela un equilibrio aceptable entre precisión y sensibilidad, siendo ideal para minimizar falsos negativos en contextos donde el costo del riesgo crediticio es alto.

El modelo XGBoost optimizado, a diferencia de lo esperado, no mejora sustancialmente respecto a su versión base y obtiene un rendimiento más bajo en casi todas las métricas clave. Su recall de 0.5633 es moderado y su F1-Score (0.5483) apenas supera al de Random Forest base (0.5362), lo que sugiere que no representa una ventaja competitiva clara en este caso.

En cuanto a la regresión logística, aunque alcanza el mayor accuracy (0.760) y la mayor precisión (0.7059), su recall (0.48) es bajo. Esto implica que, si bien clasifica con alta precisión a los clientes etiquetados como riesgosos, deja pasar a muchos que efectivamente lo son. Por tanto, podría no ser la mejor opción en escenarios donde se prioriza la detección de todos los clientes en riesgo.

En resumen, ya que el objetivo del banco es minimizar la probabilidad de otorgar créditos a clientes de alto riesgo, el modelo Random Forest optimizado ofrece la mejor solución gracias a su alto recall y F1-Score. Los demás modelos, aunque presentan buenos resultados en otras métricas, no alcanzan el mismo nivel de sensibilidad necesario para enfrentar eficazmente este problema de negocio.